# Experiment 01 — Transformer scaffold (Stage 1)

## 1. Research question

Can we stand up the **input stack** of DeepSeekFlashV4-Mini (tokenizer → embedding → RMSNorm → RoPE) with correct shapes, numerics, and a measurable tiny LM run on ≤2 GB / CPU?

## 2. Paper reference

- DeepSeek-V4.1-Flash tech report (`docs/DeepSeek_V41_Tech_Report.pdf`): Fig. 3 embedding entry; RMSNorm / RoPE discussion in architecture & KV-precision sections; Table 1 scale of V4-Flash (284B) vs what we **cannot** reproduce.
- RoPE: Su et al.; RMSNorm: Zhang & Sennrich (2019) — both used in DeepSeek stacks.

## 3. Mathematical explanation

**Embedding.** \(E \in \mathbb{R}^{V \times d}\), \(x_t = E_{z_t}\).

**RMSNorm.**
$$
\mathrm{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon},\quad
y = \frac{x}{\mathrm{RMS}(x)} \odot \gamma
$$

**RoPE** (complex form on pairs):
$$
\tilde{q}_t = q_t \, e^{i t \theta},\qquad \theta_i = \mathrm{base}^{-2i/d}
$$


In [ ]:
import sys
from pathlib import Path
PKG = Path('..').resolve()
sys.path.insert(0, str(PKG))
import torch
from tokenizer import CharTokenizer
from model.embedding import TokenEmbedding
from model.normalization import RMSNorm
from model.rope import apply_rope, rope_freqs
print('stage-1 imports OK')


## 4. Tensor shapes

| Tensor | Shape |
|--------|-------|
| tokens | `[B, T]` |
| emb out | `[B, T, d]` |
| q/k for RoPE | `[B, H, T, d_h]` |
| RoPE freqs | `[T, d_h/2]` complex |


In [ ]:
B, T, d, H, dh = 2, 8, 64, 4, 16
tok = CharTokenizer.from_text('hello world once upon a time')
emb = TokenEmbedding(tok.vocab_size, d)
norm = RMSNorm(d)
ids = torch.tensor([tok.encode('hello wo')[:T] for _ in range(B)])
x = emb(ids)
print('emb', tuple(x.shape))
print('norm', tuple(norm(x).shape))
q = torch.randn(B, H, T, dh)
print('rope', tuple(apply_rope(q).shape), 'freqs', tuple(rope_freqs(dh, T, q.device).shape))


## 5. Naive implementation

Complex multiply RoPE in `model/rope.py` (`apply_rope`).

## 6. Optimized implementation

Deferred (fused RoPE-attention kernels are production-only). Stage-1 keeps the readable complex form.

## 7. Correctness test


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, str(PKG / 'tests' / 'test_rope.py')], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
assert r.returncode == 0


## 8. Training experiment

Run `training/stage01_train.py` (Level-0 char corpus; temporary inline attention marked `[EXPERIMENTAL]`).


In [ ]:
import subprocess, sys, json
r = subprocess.run([sys.executable, str(PKG / 'training' / 'stage01_train.py')], capture_output=True, text=True)
print(r.stdout[-2000:] if len(r.stdout)>2000 else r.stdout)
print(r.stderr)
assert r.returncode == 0
metrics = json.loads((PKG.parents[0] / 'results' / 'stage01' / 'metrics.json').read_text())
metrics


## 9. Benchmark / 10. Result / 11. Interpretation

See printed `metrics.json`. Expect train/val loss to drop on the tiny repeated corpus; absolute numbers are **not** comparable to Flash.

## 12. Production difference

Cannot reproduce 284B parameters, BPE vocab, CSA–HCA, MoE at production width, or cluster KV replay on a 2 GB GPU. Stage-1 only verifies the **input + position** primitives.
